# GrandNexus v7.7.3 — check of the repair of logbook 90 (optional, 1 call)
The translator now retries transient API errors (unavailable, overloaded, rate limit, timeout) with growing waits, and the
diagnosis tool counts the facts of failed readings in its gate. This notebook checks, on Colab's own SDK:
**cell 2** one real call goes through the retry layer (1 call, under $0.01); **cell 3** the 20 offline tests and the boot
profiles (0 calls); **cell 4** the gate of logbook 89 recomputed from its archive (0 calls).

Expected: cell 2 prints `reply received: True` and `retries during the call: 0`; cell 3 prints 20 `ok` lines and four PASS;
cell 4 prints `repaired gate: ... -> FAIL (50/120 readings failed ...)` and `of which superseded ... 3`.
Optional: the next measurement notebook repeats this pre-flight. Run the cells in order; if the Colab assistant offers to
rewrite a cell, decline. Send back the output of cells 2 to 5 and the zip named in cell 5 (its full name).

## 1. Get exactly v7.7.3

In [ ]:
# 1. Get EXACTLY GrandNexus v7.7.3 (never another zip; stop if it is not found)
import os, glob, fnmatch, subprocess, zipfile, shutil, time
VERSION, ZIP_PATTERN = 'v7.7.3', 'GrandNexus_v7.7.3*.zip'
CONTENT, WORK = '/content', '/content/GrandNexus'
MYDRIVE = '/content/drive/MyDrive'

def gn_version(path):
    """(git describe, has the v7.7.3 repair) of a GrandNexus folder, or (None, False)."""
    if not os.path.isdir(os.path.join(path, '.git')):
        return None, False
    d = subprocess.run('git describe --tags', shell=True, cwd=path, capture_output=True, text=True).stdout.strip()
    tr = os.path.join(path, 'grandnexus', 'llm_interface', 'translator_v5.py')
    diag = os.path.join(path, 'tools', 'living_diag.py')
    ok = all(os.path.exists(x) for x in (tr, diag, os.path.join(path, 'tools', 'colab_save.py'),
                                          os.path.join(path, 'tests_lab', 'test_api_retry.py'))) and \
         'def _create' in open(tr, encoding='utf-8').read() and '--regate' in open(diag, encoding='utf-8').read()
    return d, ok

try:
    from google.colab import drive; drive.mount(os.path.join(CONTENT, 'drive'))
except Exception as e:
    print('Drive not mounted:', e)

found = glob.glob(os.path.join(MYDRIVE, 'GrandNexus', ZIP_PATTERN)) + glob.glob(os.path.join(MYDRIVE, ZIP_PATTERN))
if not found:
    for root, dirs, names in os.walk(MYDRIVE):
        if root[len(MYDRIVE):].count(os.sep) >= 3:
            dirs[:] = []
        found += [os.path.join(root, n) for n in names if fnmatch.fnmatch(n, ZIP_PATTERN)]
found.sort(key=os.path.getmtime)
if found:
    source = found[-1]
else:
    print(f'{ZIP_PATTERN} not found in My Drive: upload it now (any other zip is refused).')
    try:
        from google.colab import files
        up = next(iter(files.upload()))
    except Exception as e:
        raise SystemExit(f'STOP: no {ZIP_PATTERN} on the Drive and no upload possible ({e}).')
    if not fnmatch.fnmatch(os.path.basename(up), ZIP_PATTERN):
        raise SystemExit(f'STOP: {up} is not {ZIP_PATTERN}.')
    source = os.path.join(CONTENT, up)
print('release zip:', source)
print('sha256:', subprocess.run(['sha256sum', source], capture_output=True, text=True).stdout.split()[0][:16])

# an existing /content/GrandNexus of another version is moved aside, never reused, never deleted
d, ok = gn_version(WORK)
if os.path.exists(WORK) and not ((d or '').startswith(VERSION) and ok):
    aside = WORK + '_aside_' + time.strftime('%Y%m%d_%H%M%S')
    shutil.move(WORK, aside); print(f'existing {WORK} ({d}) moved aside to {aside}')
if not os.path.exists(WORK):
    z = zipfile.ZipFile(source)
    top = {n.split('/')[0] for n in z.namelist() if n.strip('/')}
    z.extractall(CONTENT if top == {'GrandNexus'} else WORK)
    print('extracted')
subprocess.run('git checkout -q main', shell=True, cwd=WORK)
d, ok = gn_version(WORK)
print('git describe:', d, '| v7.7.3 repair present:', ok)
print(subprocess.run('git log --oneline -1 && git branch', shell=True, cwd=WORK, capture_output=True, text=True).stdout)
if not ((d or '').startswith(VERSION) and ok):
    raise SystemExit(f'STOP: {WORK} is {d}, not {VERSION}. Do not run the next cells.')
print(f'OK: {VERSION}')


## 2. Packages, key, pre-flight through the retry layer (1 call)

In [ ]:
# 2. Packages, key, pre-flight check (1 call, through the translator and its retry layer)
import subprocess, sys, os
WORK = '/content/GrandNexus'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic', 'clingo', 'sympy', 'networkx', 'psutil'])
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, WORK)
import anthropic
import grandnexus.llm_interface.translator_v5 as T
print('SDK', anthropic.__version__, '| classes present:',
      {n: hasattr(anthropic, n) for n in ('ServiceUnavailableError', 'OverloadedError', 'RateLimitError', 'APIConnectionError')})
print('retry waits (s):', T.DEFAULT_RETRY_WAITS)
tr = T.Translator(model='claude-sonnet-5-5')
reply = tr._call(T.SYSTEM_TRANSLATE, 'TEXT:\nAll dogs bark. Rex is a dog.\n\nQUESTION (or statement to verify):\nDoes Rex bark?', 'preflight')
r = tr.receipts[-1]
print('reply received:', bool(reply.strip()), '| id:', r.id, '| model:', r.model)
print(f'usage: input {r.tokens_in} | output {r.tokens_out} | cache write {r.cache_write} | cache read {r.cache_read}')
print('retries during the call:', len(tr.retry_log), tr.retry_log or '')


## 3. Offline tests and boot profiles (0 calls)

In [ ]:
# 3. Offline tests and boot profiles (0 calls)
import subprocess, sys, glob, os
WORK = '/content/GrandNexus'
bad = []
for t in sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'test_*.py'))):
    r = subprocess.run([sys.executable, '-W', 'ignore', t], cwd=WORK, capture_output=True, text=True)
    last = (r.stdout.strip().splitlines() or ['(no output)'])[-1]
    ok = r.returncode == 0 and not last.startswith('FAIL') and 'Traceback' not in r.stderr
    print(f"{'ok ' if ok else 'BAD'} {os.path.basename(t):40} {last[:70]}")
    if not ok:
        bad.append(t)
r = subprocess.run([sys.executable, '-W', 'ignore', 'tools/boot_profiles.py'], cwd=WORK, capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if 'PASS' in l or 'FAIL' in l))
if bad:
    raise SystemExit(f'STOP: {len(bad)} offline test(s) failed; send this output.')


## 4. The gate of logbook 89, recomputed from its archive (0 calls)

In [ ]:
# 4. Replay of the archived gate (0 calls): every fact counted, failed readings as not read, moves from the world
import subprocess, sys, glob, os
WORK = '/content/GrandNexus'
arch = sorted(glob.glob(WORK + '/tests_lab/resultats/living/living_diag_L1_2026-11-13_*.json'))
if not arch:
    raise SystemExit('STOP: the archived diagnosis run of 2026-11-13 is not in the repository.')
r = subprocess.run([sys.executable, '-W', 'ignore', 'tools/living_diag.py', '--regate', arch[0]], cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr[-300:])


## 5. Save to Drive

In [ ]:
# 5. Save to Drive (the repository's own tool: it zips the repository it belongs to and checks the results are inside)
import subprocess, sys
WORK = '/content/GrandNexus'
r = subprocess.run([sys.executable, 'tools/colab_save.py', '--tag', 'check773', '--expect', 'tests_lab/resultats/living/living_regate_*.json'],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])
